# Assignment -- Cedar Grove Public Library: Checkouts

**5 problems, basic -> medium.** Same core skills as class (pandas: loading, cleaning,
grouping; `requests`: calling a real public API and parsing JSON) applied to a brand-new
scenario and dataset.

**Estimated time:** 45-60 minutes

## The scenario

Cedar Grove Public Library tracks every book checkout in `data/checkouts.csv`: who
checked out which book, when it was due back, when (if ever) it was returned, and any
late fee charged. The head librarian has five questions.

## Setup

```bash
pip install -r requirements.txt
python scripts/generate_checkouts_data.py   # only if data/checkouts.csv isn't already there
jupyter lab Assignment_Starter.ipynb
```

Problem 4 calls a real public API ([Open Library](https://openlibrary.org/)) and needs an
open internet connection; its function includes a fallback so the assignment stays
completable even if that call fails.

Each problem names the exact variable you need to produce, and most have a small
"check yourself" cell with `assert` statements right after.

In [ ]:
import pandas as pd
import numpy as np
import requests

pd.set_option("display.max_columns", 20)

---
## Problem 1 (basic) -- Load and get oriented

Load `data/checkouts.csv` into `checkouts_df`, parsing `checkout_date`, `due_date`, and
`return_date` as real dates. Then answer: how many checkouts are there in total, and how
many have never been returned (i.e. `return_date` is missing)? Store these two numbers as
`n_total_checkouts` and `n_still_checked_out`.

In [ ]:
# TODO: load data/checkouts.csv, parsing the three date columns
checkouts_df = ...

# TODO: how many rows total, and how many are missing return_date?
n_total_checkouts = ...
n_still_checked_out = ...

print(f"{n_total_checkouts} total checkouts, {n_still_checked_out} still checked out")

In [ ]:
# Check yourself
assert n_total_checkouts == len(checkouts_df)
assert n_still_checked_out == checkouts_df["return_date"].isna().sum()
assert n_still_checked_out < n_total_checkouts
print("Looks good.")

---
## Problem 2 (basic-medium) -- Clean the data, the right way for each column

A missing `return_date` here does **not** mean bad data -- it means the book is still
checked out, which is a completely normal, valid state. So this time, don't drop those
rows or invent a fake return date for them; instead, capture "returned or not" explicitly.

Build `checkouts_clean` from `checkouts_df` with:

- a new boolean column `is_returned`, `True` where `return_date` is present and `False`
  where it's missing
- `late_fee` missing filled with `0` (missing here means no fee was ever charged -- either
  the book came back on time, or a librarian waived the fee)

In [ ]:
checkouts_clean = checkouts_df.copy()

# TODO: add an is_returned boolean column (True where return_date is present)
checkouts_clean["is_returned"] = ...

# TODO: fill missing late_fee with 0
checkouts_clean["late_fee"] = ...

checkouts_clean.head()

In [ ]:
# Check yourself
assert checkouts_clean["late_fee"].isna().sum() == 0
assert checkouts_clean["is_returned"].dtype == bool
assert checkouts_clean["is_returned"].sum() == checkouts_clean["return_date"].notna().sum()
print("Looks good:", checkouts_clean["is_returned"].value_counts().to_dict())

---
## Problem 3 (medium) -- Which genre racks up the most late fees?

Using only **returned** books (`is_returned == True`), compute the average `late_fee` per
`genre`, sorted from highest to lowest, as `avg_late_fee_by_genre`.

In [ ]:
# TODO: filter to returned books only, then average late_fee by genre, sorted descending
returned_only = ...
avg_late_fee_by_genre = ...
avg_late_fee_by_genre

In [ ]:
# Check yourself
assert len(avg_late_fee_by_genre) == checkouts_clean["genre"].nunique()
assert avg_late_fee_by_genre.is_monotonic_decreasing
print("Looks good -- worst genre for late fees:", avg_late_fee_by_genre.idxmax())

---
## Problem 4 (basic-medium) -- Look up each book with a real public API

The library's own data doesn't say who wrote each book or when it was first published.
[Open Library](https://openlibrary.org/) has a free, no-API-key search endpoint that does:

`https://openlibrary.org/search.json?q={title}`

The response's `"docs"` list holds the matches; the first one is normally the best match.
Each doc has `author_name` (a **list** of strings) and `first_publish_year`.

Write `get_book_facts(title)`, returning `{"author": ..., "first_publish_year": ...}` for
the top match. Wrap the request in a `try`/`except` and fall back to
`BACKUP_BOOK_FACTS[title]` (given below) if the call fails or the response looks wrong --
same resilience pattern as the real-API-with-a-fallback from class.

In [ ]:
# Known-correct backup facts (classroom fallback only, used if the live API call fails).
BACKUP_BOOK_FACTS = {
    "Pride and Prejudice": {"author": "Jane Austen", "first_publish_year": 1813},
    "To Kill a Mockingbird": {"author": "Harper Lee", "first_publish_year": 1960},
    "The Great Gatsby": {"author": "F. Scott Fitzgerald", "first_publish_year": 1925},
    "The Catcher in the Rye": {"author": "J. D. Salinger", "first_publish_year": 1951},
    "1984": {"author": "George Orwell", "first_publish_year": 1949},
    "Brave New World": {"author": "Aldous Huxley", "first_publish_year": 1932},
    "Frankenstein": {"author": "Mary Shelley", "first_publish_year": 1818},
    "Jane Eyre": {"author": "Charlotte Bronte", "first_publish_year": 1847},
    "Moby Dick": {"author": "Herman Melville", "first_publish_year": 1851},
    "The Hobbit": {"author": "J. R. R. Tolkien", "first_publish_year": 1937},
    "War and Peace": {"author": "Leo Tolstoy", "first_publish_year": 1869},
    "Crime and Punishment": {"author": "Fyodor Dostoevsky", "first_publish_year": 1866},
}

OPEN_LIBRARY_API = "https://openlibrary.org/search.json"

In [ ]:
def get_book_facts(title):
    # TODO: GET OPEN_LIBRARY_API with params={"q": title}, raise_for_status(), take
    # response.json()["docs"][0], and return {"author": ..., "first_publish_year": ...}
    # (author_name is a list -- use its first item). On any RequestException, KeyError,
    # or IndexError, fall back to BACKUP_BOOK_FACTS[title] instead of crashing.
    ...

get_book_facts("1984")

Now call it for every distinct title in the library's catalog and assemble `book_facts_df`,
indexed by `book_title`, with columns `author` and `first_publish_year`.

In [ ]:
# TODO: call get_book_facts for every unique book_title in checkouts_clean and
# build book_facts_df indexed by book_title
records = {}
# your loop here

book_facts_df = pd.DataFrame(records).T
book_facts_df

In [ ]:
# Check yourself
assert len(book_facts_df) == checkouts_clean["book_title"].nunique()
assert set(["author", "first_publish_year"]).issubset(book_facts_df.columns)
print("Looks good:", book_facts_df.shape)

---
## Problem 5 (medium) -- Which author costs the library the most in late fees?

Merge `book_facts_df` into `checkouts_clean` (on `book_title`), then compute total
`late_fee` collected **per author**, sorted descending, as `late_fee_by_author`.

In [ ]:
# TODO: merge checkouts_clean with book_facts_df on book_title (book_facts_df's
# index is the title, so you'll need it as a column first -- see reset_index()),
# then total late_fee by author, sorted descending
checkouts_with_author = ...
late_fee_by_author = ...
late_fee_by_author

In [ ]:
# Check yourself
assert len(checkouts_with_author) == len(checkouts_clean)
assert late_fee_by_author.is_monotonic_decreasing
print("Looks good -- costliest author:", late_fee_by_author.idxmax())

---
## Submission

Save this notebook with your name in the filename (e.g. `Assignment_JaneDoe.ipynb`) with
all cells run top to bottom.